# 证据复核与指定组合重测

填写下方 `RESULT_DIR` 为你本地某一批的完整结果目录。本公开包不包含全部原始运行数据。前面的工作台与用量汇总只读本地文件；准备计划与真实运行分开执行。


In [ ]:
from pathlib import Path
import json, os, sys, subprocess
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools" / "review_workbench.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("请在VS Code打开完整项目文件夹")
RESULT_DIR = PROJECT_ROOT / "evaluation" / "expanded_campaigns" / "my_evaluation_v1" / "results" / "REPLACE_WITH_BATCH_FOLDER"
CHILD_ENV = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
if not RESULT_DIR.is_dir():
    raise FileNotFoundError("请将RESULT_DIR改为真实结果目录：" + str(RESULT_DIR))
print("项目：", PROJECT_ROOT)
print("待整理的实验：", RESULT_DIR)

## 生成离线复核工作台
完成后，在文件管理器中打开打印的`review_workbench.html`。可以在页面直接填写状态、复核人、证据与适配判断，点击导出，把下载的JSON放回此实验目录并命名为`human_reviews.json`，然后重建矩阵。

页面集中展示每轮回答、原文、匹配提示和截图链接。精确片段命中不证明引用归属或结论成立；材料哈希不一致时不会做原文匹配。若材料为PDF或页面来源卡片未进答案正文，需要打开原文件/截图补充核查。

In [ ]:
built = subprocess.run([sys.executable, str(PROJECT_ROOT / "tools/review_workbench.py"), str(RESULT_DIR),
                        "--project-root", str(PROJECT_ROOT)], env=CHILD_ENV,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
print(built.stdout, built.stderr)
if built.returncode:
    raise RuntimeError("工作台生成失败，请保留上方错误输出")
print("请打开：", RESULT_DIR / "review_workbench.html")

## 查看已经记录的用量
只读取已有结果。表单编辑计时为近似经过时间，旧复核没有计时数据。控制模型/Persona tokens分别统计；被测产品内部用量未知。不从不同发布条件的结果计算效率提升。

In [ ]:
cost = subprocess.run([sys.executable, str(PROJECT_ROOT / "tools/cost_summary.py"), str(RESULT_DIR)],
                      env=CHILD_ENV, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(cost.stdout, cost.stderr)
if cost.returncode:
    raise RuntimeError("用量整理失败")

## 更新矩阵（不调用产品）
默认不执行；保存复核JSON后设`REBUILD_REPORT=True`。

In [ ]:
REBUILD_REPORT = False
RUNNER_DIR = PROJECT_ROOT / "evaluator"
if REBUILD_REPORT:
    rebuilt = subprocess.run([sys.executable, "runner.py", "--rebuild", str(RESULT_DIR)],
                             cwd=str(RUNNER_DIR), env=CHILD_ENV, capture_output=True,
                             text=True, encoding="utf-8", errors="replace")
    print(rebuilt.stdout, rebuilt.stderr)
    if rebuilt.returncode:
        raise RuntimeError("重建失败，检查复核状态/复核人/证据定位")
else:
    print("待复核文件：", RESULT_DIR / "human_reviews.json")

## 准备精确重测计划（只生成配置，不调用API）
`execution_errors`只选执行异常；`review_failed`只选人工任务/来源复核失败；`unreviewed`会重跑未复核回答，通常先复核已有答案更省成本。

不会把组合扩成整道题的所有Persona或两个产品。沿用原材料哈希，另建实验目录。若已重新发布、换模型/检索设置，应补记新配置。`RERUN_OPERATION_MODE="rules"`用于测试新控制方式，不是严格复现旧控制模式；要复现原控制方式可设为`None`。

In [ ]:
PREPARE_RERUN = False
RERUN_SCOPE = "execution_errors"
RERUN_OPERATION_MODE = "rules"  # None为沿用源实验模式
RERUN_CONFIG = None
if PREPARE_RERUN:
    command = [sys.executable, str(PROJECT_ROOT / "tools/prepare_rerun.py"), str(RESULT_DIR),
               "--scope", RERUN_SCOPE, "--project-root", str(PROJECT_ROOT)]
    if RERUN_OPERATION_MODE is not None:
        command += ["--operation-mode", RERUN_OPERATION_MODE]
    prepared = subprocess.run(command, env=CHILD_ENV, capture_output=True,
                              text=True, encoding="utf-8", errors="replace")
    if prepared.returncode:
        print(prepared.stdout, prepared.stderr)
        raise RuntimeError("准备失败：需要原config.json，且本地材料版本必须相同")
    result = json.loads(prepared.stdout)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    if result["config_written"]:
        RERUN_CONFIG = Path(result["config_path"])
        preview = subprocess.run([sys.executable, "runner.py", "--plan", str(RERUN_CONFIG)],
                                 cwd=str(RUNNER_DIR), env=CHILD_ENV, capture_output=True,
                                 text=True, encoding="utf-8", errors="replace")
        print(preview.stdout, preview.stderr)
        if preview.returncode:
            raise RuntimeError("重测计划不合法")
else:
    print("未准备重测；已有答案可以直接复核。")

## 主动启动重测（这里才会调用产品及所需API）
先检查计划、确认发布和配置状态，才设`RUN_RERUN=True`。默认不执行。后台日志可用下一格查看。

In [ ]:
RUN_RERUN = False
if RUN_RERUN:
    if RERUN_CONFIG is None:
        raise RuntimeError("先生成非空重测配置并检查计划")
    if "RERUN_JOB" in globals() and RERUN_JOB.poll() is None:
        raise RuntimeError("已有重测任务运行中")
    cfg = json.loads(RERUN_CONFIG.read_text(encoding="utf-8"))
    if Path(cfg["output_dir"]).exists():
        raise RuntimeError("此配置已运行；请重新准备配置建立新实验")
    RERUN_LOG = RERUN_CONFIG.with_suffix(".log")
    with RERUN_LOG.open("w", encoding="utf-8") as log:
        RERUN_JOB = subprocess.Popen([sys.executable, "runner.py", "--config", str(RERUN_CONFIG)],
                                    cwd=str(RUNNER_DIR), env=CHILD_ENV, stdout=log, stderr=subprocess.STDOUT)
    RERUN_LOG_OFFSET = 0
    print("已启动，结果：", cfg["output_dir"])
else:
    print("未启动重测，没有调用产品或API。")

In [ ]:
if "RERUN_JOB" not in globals():
    print("没有运行中的重测任务。")
else:
    with RERUN_LOG.open("r", encoding="utf-8", errors="replace") as log:
        log.seek(globals().get("RERUN_LOG_OFFSET", 0));print(log.read());RERUN_LOG_OFFSET=log.tell()
    print("进程返回码：", RERUN_JOB.poll())